In [33]:
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import ttest_1samp

In [7]:
data_df = pl.scan_csv(
    "../data/payments.csv",
).collect()
data_df.head()


date,vwretd,ewretd,totval,div_tot_paid,div_tot_declared,div_tot_recorded
str,f64,f64,f64,f64,str,f64
"""31dec1925""",null,null,2.7487e7,0.0,null,null
"""02jan1926""",0.005689,0.009516,2.7600e7,0.0,null,null
"""04jan1926""",0.000706,0.00578,2.7578e7,0.0,null,null
"""05jan1926""",-0.004821,-0.001927,2.7530e7,0.0,null,1141.0
"""06jan1926""",-0.000423,0.001182,2.7619e7,0.0,null,100.0


- date: trading day
- vwretd: value-weighted portfoli return
- ewretd: equal-weighted portfolio return
- totval: total market cap of all US common stocks
- div_tot_paid: the total $ of ordinary cash dividends **paid** on that day
- div_tot_declared: the total $ of ordinary cash dividends **declared** on that day
- div_tot_recorded: the total $ of ordinary cash dividends **recorded** on that day

In [ ]:
# format the date
data_df = data_df.with_columns(
    pl.col("date")
    .str.strip_chars()
    .str.to_date(format="%d%b%Y", strict=False)
    .alias("date")
)
data_df.head()


date,vwretd,ewretd,totval,div_tot_paid,div_tot_declared,div_tot_recorded
date,f64,f64,f64,f64,str,f64
1925-12-31,null,null,2.7487e7,0.0,null,null
1926-01-02,0.005689,0.009516,2.7600e7,0.0,null,null
1926-01-04,0.000706,0.00578,2.7578e7,0.0,null,null
1926-01-05,-0.004821,-0.001927,2.7530e7,0.0,null,1141.0
1926-01-06,-0.000423,0.001182,2.7619e7,0.0,null,100.0


## Question 1
create "abnormal dividend yield" column

In [ ]:
def construct_abnormal_dividend_yield(
    data_df: pl.DataFrame, div_col: str
) -> pl.DataFrame:
    return (
        data_df.sort("date")
        .with_columns(
            pl.col(div_col).rolling_sum(window_size=2).alias(f"{div_col}_2"),
            pl.col(div_col)
            .rolling_mean(window_size=233, min_samples=233)
            .shift(20)
            .alias(f"avg_{div_col}"),
        )
        .with_columns(
            (pl.col(f"{div_col}_2") / pl.col(f"avg_{div_col}")).alias(
                f"abnormal_dividend_yield({div_col})"
            )
        )
    )


In [38]:
data_df2 = construct_abnormal_dividend_yield(data_df, "div_tot_paid")

In [40]:
# summary statistics
data_df2["abnormal_dividend_yield(div_tot_paid)"].describe()

statistic,value
str,f64
"""count""",24794.0
"""null_count""",252.0
"""mean""",2.107223
"""std""",3.502997
"""min""",-9.8360e-15
"""25%""",0.196158
"""50%""",0.682675
"""75%""",2.331666
"""max""",50.957995


## Question 2
regress the value-weighted and equal-weighted market returns against the abnormal dividend yields. What is the t-statistics on this coefficient?

In [ ]:
regression_data = data_df2[
    ["vwretd", "ewretd", "abnormal_dividend_yield(div_tot_paid)"]
].drop_nulls()
X = sm.add_constant(
    regression_data[["abnormal_dividend_yield(div_tot_paid)"]].to_numpy()
)

In [42]:
# regression 1: vwretd ~ abnormal_dividend_yield
y = regression_data["vwretd"].to_numpy()
model1 = sm.OLS(y, X).fit(missing="drop")
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     12.80
Date:                Mon, 09 Mar 2026   Prob (F-statistic):           0.000348
Time:                        21:05:50   Log-Likelihood:                 77238.
No. Observations:               24794   AIC:                        -1.545e+05
Df Residuals:                   24792   BIC:                        -1.545e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0003   7.96e-05      3.420      0.0

In [45]:
model1.rsquared

np.float64(0.0005158484338828595)

In [46]:
# model 2: ewretd ~ abnormal_dividend_yield
y = regression_data["ewretd"].to_numpy()
model2 = sm.OLS(y, X).fit(missing="drop")
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     12.28
Date:                Mon, 09 Mar 2026   Prob (F-statistic):           0.000458
Time:                        21:06:35   Log-Likelihood:                 77584.
No. Observations:               24794   AIC:                        -1.552e+05
Df Residuals:                   24792   BIC:                        -1.551e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0007   7.85e-05      8.839      0.0

In [47]:
model2.rsquared

np.float64(0.0004952100565867035)

## Question 3
quintile

In [ ]:
def construct_quintile_analysis(data_df: pl.DataFrame, group_col: str) -> pl.DataFrame:
    quintile_df = data_df.drop_nulls(subset=["vwretd", group_col]).with_columns(
        pl.col(group_col)
        .qcut(5, labels=["q1", "q2", "q3", "q4", "q5"])
        .alias("quintile"),
    )

    results = []
    for quintile in ["q1", "q2", "q3", "q4", "q5"]:
        returns = quintile_df.filter(pl.col("quintile") == quintile)[
            "vwretd"
        ].to_numpy()

        mean_ret = returns.mean()
        mean_ady = (
            quintile_df.filter(pl.col("quintile") == quintile)[group_col]
            .to_numpy()
            .mean()
        )
        n_obs = len(returns)

        # Perform one-sample t-test (H0: mean = 0)
        t_stat, p_value = ttest_1samp(returns, 0)

        results.append(
            {
                "quintile": quintile,
                "mean_vwretd": mean_ret,
                "mean_abnormal_dividend_yield": mean_ady,
                "n_obs": n_obs,
                "t_stat": t_stat,
                "p_value": p_value,
            }
        )

    return pl.DataFrame(results)

In [56]:
construct_quintile_analysis(data_df2, "abnormal_dividend_yield(div_tot_paid)")

quintile,mean_vwretd,mean_abnormal_dividend_yield,n_obs,t_stat,p_value
str,f64,f64,i64,f64,f64
"""q1""",0.000164,0.041355,4959,0.969823,0.332182
"""q2""",0.000235,0.270714,4959,1.547281,0.121859
"""q3""",0.000309,0.71563,4958,2.094182,0.036294
"""q4""",0.000581,1.904002,4959,4.227214,0.000024
"""q5""",0.000804,7.604132,4959,5.228334,1.7807e-7


## Question 4
rerun the analysis in question 3 but use the recorded dividends instead of the paid dividends.

In [57]:
data_df3 = construct_abnormal_dividend_yield(data_df, "div_tot_recorded")
construct_quintile_analysis(data_df3, "abnormal_dividend_yield(div_tot_recorded)")

quintile,mean_vwretd,mean_abnormal_dividend_yield,n_obs,t_stat,p_value
str,f64,f64,i64,f64,f64
"""q1""",0.000446,0.285743,2203,1.964125,0.049641
"""q2""",0.000257,0.798538,2202,1.133276,0.257222
"""q3""",0.000298,1.502855,2203,1.316487,0.188148
"""q4""",0.000522,2.656888,2202,2.365537,0.01809
"""q5""",0.000644,5.206729,2203,2.857208,0.004314


## Question 5: Interpretation of Findings

**Key Results Summary:**

**Using Paid Dividends (Question 3):**
- Clear monotonic relationship: higher abnormal dividend yield → higher returns
- Q1 (lowest): 0.0164% (not significant, p=0.332)
- Q5 (highest): 0.0804% (highly significant, p<0.001)
- t-statistic on regression coefficient: 3.577 (p<0.001)

**Using Recorded Dividends (Question 4):**
- Similar but weaker pattern
- Q1: 0.0446% (marginally significant, p=0.050)
- Q5: 0.0644% (significant, p=0.004)
- Less pronounced difference between quintiles

**Interpretation:**

The results are puzzling from an efficient markets perspective. We observe **positive, statistically significant returns on high dividend payment days**, even though:

1. **No new information**: Payment dates occur ~43 days after announcement and ~22 days after ex-date. All economically meaningful news and tax implications have already been resolved.

2. **Pure cash transfer**: The payment date is simply when cash is disbursed—it's a mechanical event with no fundamental information content.

3. **Predictable timing**: These payments are scheduled and known in advance.

**Possible Explanations:**

1. **Price pressure / liquidity effect**: Large dividend payments may create temporary buying pressure as investors reinvest cash, driving up prices on payment days.

2. **Institutional rebalancing**: Mutual funds and other institutions may systematically rebalance portfolios around dividend receipts, creating predictable demand.

3. **Tax-loss harvesting / clientele effects**: Even though ex-date resolves tax consequences, actual cash receipt may trigger portfolio adjustments.

4. **Measurement issues**: The weaker results with recorded dividends (which are earlier in the process) suggest the effect is concentrated around actual cash disbursement, not information flow.

The fact that returns are higher for **paid** dividends than **recorded** dividends supports a mechanical/liquidity explanation rather than an information-based one, since recorded dividends occur earlier in the process when information effects should be stronger.